https://lamport.azurewebsites.net/tla/tutorial/intro.html

EXPORT CLASSPATH=tla2tools.jar
java tla2sany.SANY -help  # The TLA⁺ parser
java tlc2.TLC -help       # The TLA⁺ model checker
java tlc2.REPL            # Enter the TLA⁺ REPL
java pcal.trans -help     # The PlusCal-to-TLA⁺ translator
java tla2tex.TLA -help    # The TLA⁺-to-LaTeX translator
java tla2sany.xml.XMLExporter -help # Export TLA⁺ parse tree as XML


In [24]:
%%file /tmp/mytest.tla
------ MODULE mytest -----------

Foo == 3

(*
--algorithm AnyName {
   variable x = {"a","b"} ;  
   {  
     x := x \cup {"c"} ;
     print x
   }
}
*)

===============================

Overwriting /tmp/mytest.tla


-o makes it not validate against a schema that is no longer being served. It is not output. Insanity. Complete derangement.

In [25]:
! java -cp ~/Downloads/tla2tools.jar tla2sany.xml.XMLExporter -o /tmp/mytest.tla

Semantic processing of module mytest
<?xml version="1.0" encoding="UTF-8" standalone="no"?><modules><RootModule>mytest</RootModule><context><entry><UID>154</UID><ModuleNode><location><column><begin>1</begin><end>31</end></column><line><begin>1</begin><end>15</end></line><filename>mytest</filename></location><uniquename>mytest</uniquename><UserDefinedOpKindRef><UID>156</UID></UserDefinedOpKindRef></ModuleNode></entry><entry><UID>156</UID><UserDefinedOpKind><location><column><begin>1</begin><end>8</end></column><line><begin>3</begin><end>3</end></line><filename>mytest</filename></location><level>0</level><uniquename>Foo</uniquename><arity>0</arity><body><NumeralNode><location><column><begin>8</begin><end>8</end></column><line><begin>3</begin><end>3</end></line><filename>mytest</filename></location><level>0</level><IntValue>3</IntValue></NumeralNode></body><params/></UserDefinedOpKind></entry></context><ModuleNodeRef><UID>154</UID></ModuleNodeRef></modules>

In [ ]:
! java -cp ~/Downloads/tla2tools.jar pcal.trans /tmp/mytest.tla && cat /tmp/mytest.tla && java -cp ~/Downloads/tla2tools.jar 

pcal.trans Version 1.11 of 31 December 2020
Labels added.
Parsing completed.
Translation completed.
New file /tmp/mytest.tla written.
New file /tmp/mytest.cfg written.
------ MODULE mytest -----------

(*
--algorithm AnyName {
   variable x = {"a","b"} ;  
   {  
     x := x \cup {"c"} ;
     print x
   }
}
*)
\* BEGIN TRANSLATION (chksum(pcal) = "173b0586" /\ chksum(tla) = "461a4402")
VARIABLES x, pc

vars == << x, pc >>

Init == (* Global variables *)
        /\ x = {"a","b"}
        /\ pc = "Lbl_1"

Lbl_1 == /\ pc = "Lbl_1"
         /\ x' = (x \cup {"c"})
         /\ PrintT(x')
         /\ pc' = "Done"

(* Allow infinite stuttering to prevent deadlock on termination. *)
Terminating == pc = "Done" /\ UNCHANGED vars

Next == Lbl_1
           \/ Terminating

Spec == Init /\ [][Next]_vars

Termination == <>(pc = "Done")

\* END TRANSLATION 



In [21]:
import ast


def expr(e : ast.expr):
    match e:
        case ast.Name(id=name):
            return name
        case ast.Constant(value=value):
            return repr(value)
        case _:
            raise ValueError(f"Unsupported expression: {ast.dump(e)}")

def stmt(e : ast.stmt):
    match e:
        case ast.Assign(targets=[ast.Name(id=var)], value=value):
            return f"{var} := {expr(value)}"
        case ast.Expr(value=ast.Call(func=ast.Name(id='print'), args=[arg])):
            return f"print {expr(arg)}"
        case _:
            raise ValueError(f"Unsupported statement: {ast.dump(e)}")


def to_pluscal(mod):
    match mod:
        case ast.Module(body=body):
            return "{\n" + "\n".join(stmt(e) for e in body) + "\n}"
        case _:
            raise ValueError("Expected ast.Module")
    
print(to_pluscal(ast.parse("""
x = 3
print(x)
""")))

{
x := 3
print x
}


# interurpt

In [31]:
%%file /tmp/int.s

#csrs mie, t1
    # Enable global interrupts

.global foo
foo:
csrsi   mstatus, 8      # set bit 3 (MIE)
j foo


Overwriting /tmp/int.s


In [32]:
! riscv64-unknown-elf-as -o /tmp/int.o /tmp/int.s && riscv64-unknown-elf-objdump -d /tmp/int.o


/tmp/int.o:     file format elf64-littleriscv


Disassembly of section .text:

0000000000000000 <foo>:
   0:	30046073          	csrsi	mstatus,8
   4:	ffdff06f          	j	0 <foo>


In [80]:
! nix-shell -p pkgsCross.riscv64-embedded.buildPackages.gcc --run "riscv64-none-elf-as -o /tmp/int.o /tmp/int.s && riscv64-none-elf-objdump -d /tmp/int.o"

these 9 paths will be fetched (112.73 MiB download, 621.58 MiB unpacked):
  /nix/store/vlya5cqnxgfzxjhmp7lf0b3a486qymww-newlib-riscv64-none-elf-4.5.0.20241231
  /nix/store/ackn6biybagdqkir2jgx1yhc87l5glh3-riscv64-none-elf-binutils-2.44
  /nix/store/dgfyngdwic4li5lpqam1n9ppb15br5kh-riscv64-none-elf-binutils-wrapper-2.44
  /nix/store/q4afh3d9z96iwayz856yab65346fg6g2-riscv64-none-elf-binutils-wrapper-2.44
  /nix/store/43ghl79v3hv1rz47h4p1p6cx80v5h8kz-riscv64-none-elf-gcc-14.3.0
  /nix/store/g61avw7fl52q1n3525xm8gdi6qr4djix-riscv64-none-elf-gcc-14.3.0-lib
  /nix/store/dza8g8g18glx75i2mafvzq9xna0pbbby-riscv64-none-elf-gcc-wrapper-14.3.0
  /nix/store/qjixi6h2fldxh1zvz6m1b5lwb0jh0s6y-riscv64-none-elf-nolibc-gcc-14.3.0
  /nix/store/pk2v4yb86jh16bimk5jyh86414v9bmmk-riscv64-none-elf-nolibc-gcc-14.3.0-lib
copying path '/nix/store/g61avw7fl52q1n3525xm8gdi6qr4djix-riscv64-none-elf-gcc-14.3.0-lib' from 'https://cache.nixos.org'...
copying path '/nix/store/pk2v4yb86jh16bimk5jyh86414v9bmmk-riscv64-non

Hmm. So in the generic semantics it writes to zero? And that should be ignored.

In [13]:
dir(ctx.ctx.language)
ctx.ctx.language.pspec

<Element 'processor_spec' at 0x7882b1c9a110>

In [6]:

dir(ctx.ctx)
ctx.ctx.getAllRegisters()

{<pypcode.pypcode_native.Varnode at 0x73ecb2a4aaf0>: 'CONTEXT',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a4a370>: 'pc',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a4bd20>: 'RESERVE_ADDRESS',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a4a070>: 'RESERVE',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a49b90>: 'RESERVE_LENGTH',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a49c50>: 'zero',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a4a2e0>: 'ra',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a49dd0>: 'sp',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a48ed0>: 'gp',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a49c20>: 'tp',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a49f50>: 't0',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a4a010>: 't1',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a49e60>: 't2',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a499e0>: 's0',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a49b30>: 's1',
 <pypcode.pypcode_native.Varnode at 0x73ecb2a49e90>: 'a0',
 <pypcode.pypcode_n

In [16]:
def csr_offset_size(ctx, name: str) -> tuple[int, int]:
    symbol = ctx.language.pspec.find(f".//symbol[@name='{name}']")
    if symbol is None:
        raise KeyError(f"Unknown CSR: {name}")

    space, address = symbol.attrib["address"].split(":")
    if space != "csreg":
        raise ValueError(f"{name!r} is in {space!r}, not 'csreg'")

    size = int(symbol.attrib["size"], 0)
    offset = int(address, 0) * size  # Convert csreg units to byte offset

    return offset, size

csr_offset_size(ctx.ctx, "mstatus")

(6144, 8)

In [6]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode

ctx = pcode.BinaryContext("/tmp/int.o", langid="RISCV:LE:64:default")
mem = ctx.init_mem()
mem1 = ctx.execute_block(mem,0x400000 )
ctx.loader.find_symbol("foo")
print(mem1[0].memstate.csreg.to_expr().sexpr())

def high_low(addr, m : pcode.MemState):
    return smt.Extract(3,3, m.get_reg(ctx, "mstatus"))

smt.simplify(ctx.unfold(mem1[0].memstate.get_csr(ctx, "mstatus")))
#mem1[0].memstate.get_csr(ctx, "mstatus")
#smt.simplify(ctx.unfold(high_low(0, mem)))
#smt.simplify(ctx.unfold(high_low(0, mem1[0].memstate)))



(store64le (csreg state0)
           #x0000000000001800
           (bvor (select64le (csreg state0) #x0000000000001800)
                 #x0000000000000008))


Concat(csreg(state0)[6151],
       csreg(state0)[6150],
       csreg(state0)[6149],
       csreg(state0)[6148],
       csreg(state0)[6147],
       csreg(state0)[6146],
       csreg(state0)[6145],
       Extract(7, 4, csreg(state0)[6144]),
       1,
       Extract(2, 0, csreg(state0)[6144]))

In [78]:
%%file /tmp/interrupt.tla
-------- MODULE interrupt -----------

EXTENDS Integers

VARIABLES mstatus

TypeOK == mstatus \in [MIE : BOOLEAN]

Init == mstatus = [MIE |-> FALSE]
TurnOnInterrupt == mstatus' = [MIE |-> TRUE] /* foo I guess

Next == TurnOnInterrupt
Spec == Init /\ [][Next]_mstatus

===============================

Overwriting /tmp/interrupt.tla


In [ ]:
import kdrag.solvers.tla as tla

mod = tla.Module.of_file("/tmp/interrupt.tla")
mod.infer_sorts()
mod.decls["mstatus"]

AStruct_MIE_Bool

In [ ]:
from kdrag.all import *
import kdrag.solvers.tla as tla

mod = tla.Module.of_file("/tmp/interrupt.tla")
mod.infer_sorts()
mod.decls["mstatus"]
import kdrag.contrib.pcode as pcode

ctx = pcode.BinaryContext("/tmp/int.o", langid="RISCV:LE:64:default")
mem = ctx.init_mem()
mem1 = ctx.execute_block(mem,0x400000 )
ctx.loader.find_symbol("foo")
#print(mem1[0].memstate.register.to_expr().sexpr())

mstatus,mstatus1 = smt.Consts("mstatus mstatus'", kd.AStruct(MIE=smt.BoolSort()))


def high_low(addr, m : pcode.MemState):
    return ctx.unfold(kd.astruct(MIE=smt.Extract(3,3, m.get_reg(ctx, "mstatus")) == 1))

smt.simplify(ctx.unfold(high_low(0, mem)))
smt.simplify(ctx.unfold(high_low(0, mem1[0].memstate)))

#kd.prove(mod.action("TurnOnInterrupt") == smt.And(mstatus == high_low(mem), mstatus1 == high_low(mem1[0].memstate)))
smt.simplify(smt.And(mstatus == high_low(0, mem), mstatus1 == high_low(0, mem1[0].memstate)))
# Start states are the same, the end states are the same.

kd.prove(smt.Implies(mstatus == smt.simplify(high_low(0, mem)), (mstatus1 == smt.simplify(high_low(0, mem1[0].memstate))) == (mod.action("TurnOnInterrupt"))))

|= Implies(mstatus ==
        AStruct_MIE_Bool(Extract(3,
                                 3,
                                 register(state0)[2415925248]) ==
                         1),
        (mstatus' == AStruct_MIE_Bool(True)) ==
        (mstatus' == AStruct_MIE_Bool(True)))

# hourclock

Get example off of tutorial


In [ ]:
%%file /tmp/clock.s
    .global tick
    .global reset

reset:
    mov $1, %rdi
    jmp tick
tick:
    cmp $12, %rdi
    je reset
    add $1, %rdi
    jmp tick


In [ ]:
! gcc -c /tmp/clock.s -o /tmp/clock.o

In [ ]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode

ctx = pcode.BinaryContext("/tmp/clock.o")

def high_low(addr, memstate):
    i = memstate.get_reg(ctx, "RDI")
    return i

BV64 = BitVecSort(64)
@symdef
def stepclk0(i : BV64) -> BV64:
    if i >= 12:
        return BitVecVal(1,64)
    else:
        return i + 1

ctx.bisim(high_low, stepclk0, cuts=["tick"])

In [ ]:
%%file /tmp/hourclock.c

int hr = 0;

void step() {
    if (hr == 12) {
        hr = 1;
    } else {
        hr++;
    }
}




In [ ]:
%%file /tmp/hourclock.rs

struct State {
    hr : usize
}
impl State {
    fn step(&mut self) {
        if self.hr == 12 {
            self.hr = 1;
        } else {
            self.hr += 1;
        }
    }
}


In [1]:
%%file /tmp/HourClock.tla
---- MODULE HourClock ----
EXTENDS Naturals

VARIABLE hr

HCini == hr \in (1 .. 12)

HCnxt == hr' = IF hr = 12 THEN 1 ELSE hr + 1
          (* Alternately expressed as: hr' = (hr % 12) + 1 *)

HC == HCini /\ [][HCnxt]_hr
==========================

Overwriting /tmp/HourClock.tla


C and cbmc does not support Int. If I try to enable casting from Concrete intset to BitVec, could discharge maybe that cast loses no info.
verus does
Or just a flag int_as_int

Casting _to_ Int from BitVec is not problematic.

struct of config flags passed down


In [1]:
%%file /tmp/clock.s
    .global tick
    .global reset

reset:
    mov $1, %rdi
    jmp tick
tick:
    cmp $12, %rdi
    je reset
    add $1, %rdi
    jmp tick

Writing /tmp/clock.s


In [2]:
! gcc -c /tmp/clock.s -o /tmp/clock.o

In [4]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode

ctx = pcode.BinaryContext("/tmp/clock.o")

def high_low(addr, memstate):
    i = memstate.get_reg(ctx, "RDI")
    return i

BV64 = BitVecSort(64)
@symdef
def stepclk0(i : BV64) -> BV64:
    if i >= 12:
        return BitVecVal(1,64)
    else:
        return i + 1

ctx.bisim(high_low, stepclk0, cuts=["tick"])

Unexpected SP conflict


Counterexample found for bisimulation between high and low at 
 tick  --->  tick
Initial state:
 9222246136947933186 
steps to in low
 9222246136947933187 
steps to in high
 1


Exception: bisimulation failed

In [5]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode
ctx = pcode.BinaryContext("/tmp/clock.o")

def high_low(addr, memstate):
    i = memstate.get_reg(ctx, "RDI")
    return i

BV64 = smt.BitVecSort(64)
@symdef
def step(i : BV64) -> BV64:
    if i == 12:
        return BitVecVal(1,64)
    else:
        return i + 1

ctx.bisim(high_low, step, cuts=["tick"])

Unexpected SP conflict


Bisimulation proof succeeded  over all paths [('tick', 'tick')]


In [32]:
mem1[0].memstate.get_reg(ctx, "RDI")
mem1[1].memstate.get_reg(ctx, "RDI")
mem1

[SimState(memstate=MemState((let ((a!1 (ite (bvult (select64le (register state0) &RDI) #x000000000000000c)
                 #x01
                 #x00))
       (a!2 (and (bvslt #x0000000000000000 (select64le (register state0) &RDI))
                 (bvslt #x0000000000000000 (bvneg #x000000000000000c))))
       (a!3 (bvslt #x0000000000000000
                   (bvadd (select64le (register state0) &RDI)
                          (bvneg #x000000000000000c))))
       (a!5 (bvsgt #x0000000000000000
                   (bvsub (select64le (register state0) &RDI) #x000000000000000c)))
       (a!7 (= #x0000000000000000
               (bvsub (select64le (register state0) &RDI) #x000000000000000c)))
       (a!8 (bvand (bvsub (select64le (register state0) &RDI) #x000000000000000c)
                   #x00000000000000ff)))
 (let ((a!4 (ite (= #x000000000000000c
                    (bvshl #x0000000000000001 #x000000000000003f))
                 (bvslt (select64le (register state0) &RDI) #x00000000000

In [37]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode
import kdrag.solvers.tla as tla
ctx = pcode.BinaryContext("/tmp/clock.o")
mem = ctx.init_mem()
#mem1 = ctx.execute_trace_frags(mem, ctx.loader.find_symbol("tick").rebased_addr)
mem1 = ctx.execute_trace_frags(mem, cuts=["tick"])

def high_low(memstate):
    return smt.BV2Int(memstate.get_reg(ctx, "RDI"))
len(mem1[("tick", "tick")])

Unexpected SP conflict


2

In [ ]:
from kdrag.all import *
import kdrag.contrib.pcode as pcode
import kdrag.solvers.tla as tla
ctx = pcode.BinaryContext("/tmp/clock.o")
mem = ctx.init_mem()
#mem1 = ctx.execute_trace_frags(mem, ctx.loader.find_symbol("tick").rebased_addr)
trace_frags = ctx.execute_trace_frags(mem, cuts=["tick"])
mem1 = trace_frags[("tick","tick")]
def high_low(memstate):
    return smt.BV2Int(memstate.get_reg(ctx, "RDI"))

mod = tla.Module.of_file("/tmp/HourClock.tla")
mod.infer_sorts
hr = mod.Var("hr", smt.IntSort())
hr1 = tla.prime(hr)
mod.action("HCnxt")

kd.prove(ctx.unfold(
    smt.Implies(
    smt.And(
        hr <= 1000, # We need to know hr is less than INTMAX
        hr == high_low(mem), # start synchronized
        mod.action("HCnxt")),
    smt.Or(hr1 == high_low(mem1[1].memstate), 
           hr1 == high_low(mem1[0].memstate)))))

#def lockstep():
    #smt.Implies(rel, step, rel')



Unexpected SP conflict


|= Implies(And(hr <= 1000,
            hr ==
            BV2Int(Concat(Concat(Concat(Concat(Concat(Concat(Concat(register(state0)[56 +
                                        8 -
                                        0 -
                                        1],
                                        register(state0)[56 +
                                        8 -
                                        1 -
                                        1]),
                                        register(state0)[56 +
                                        8 -
                                        2 -
                                        1]),
                                        register(state0)[56 +
                                        8 -
                                        3 -
                                        1]),
                                        register(state0)[56 +
                                        8 -
                                        4 -
                                        1]),
                                        register(state0)[56 +
                                        8 -
                                        5 -
                                        1]),
                                 register(state0)[56 + 8 -
                                   6 -
                                   1]),
                          register(state0)[56 + 8 - 7 - 1])),
            hr' == If(hr == 12, 1, hr + 1)),
        Or(hr' ==
           BV2Int(1 +
                  Concat(Concat(Concat(Concat(Concat(Concat(Concat(register(state0)[56 +
                                        8 -
                                        0 -
                                        1],
                                        register(state0)[56 +
                                        8 -
                                        1 -
                                        1]),
                                        register(state0)[56 +
                                        8 -
                                        2 -
                                        1]),
                                        register(state0)[56 +
                                        8 -
                                        3 -
                                        1]),
                                        register(state0)[56 +
                                        8 -
                                        4 -
                                        1]),
                                       register(state0)[56 +
                                        8 -
                                        5 -
                                        1]),
                                register(state0)[56 + 8 -
                                  6 -
                                  1]),
                         register(state0)[56 + 8 - 7 - 1])),
           hr' == BV2Int(1)))

# verus

In [ ]:
import kdrag.solvers.tla as tla
import kdrag as kd
import kdrag.printers.rust as rust
import kdrag.printers.c as c
import kdrag.contrib.pcode as pcode
import kdrag.smt as smt

mod = tla.Module.of_file("/tmp/HourClock.tla")
#mod.declare_var("hr", smt.BitVecSort(8))
#hr = mod.Var("hr", smt.BitVecSort(8))
mod.infer_sorts() # mod.check_sorts? # needs more casting
mod.decls

#c.c_of_expr([], [], smt.simplify(mod.action("HCini", {})))
#rust.of_expr(smt.simplify(mod.action("HCini", {}))) # Int is swakward
rust.of_expr(mod.action("HCini", {})) # Lambda is awkward

NotImplementedError: ('Cannot convert quantifier to Rust expressions', Lambda(x!86, And(1 <= x!86, 12 >= x!86)))

In [ ]:
%%file /tmp/hourclock.rs

#[allow(unused_imports)]
use verus_builtin::*;
use verus_builtin_macros::*;
use vstd::prelude::*;
use vstd::{pervasive::*, *};

use verus_state_machines_macros::state_machine;
use verus_state_machines_macros::tokenized_state_machine;


verus! {
  state_machine!{
     Clock {
        fields {
        pub hr : int,
        }

        init!{
            initialize() {
            init hr = 1;
            }
        }
        transition!{
            HCnxt() {
                update hr = if pre.hr == 12 { 1 } else { pre.hr + 1 };
            }
        }
        #[invariant]
        pub fn is_typeof(&self) -> bool {
            self.hr >= 1 && self.hr <= 12
        }

        #[inductive(initialize)]
        fn initialize_inductive(post: Self) { }
       
        #[inductive(HCnxt)]
        fn HCnxt_inductive(pre: Self, post: Self) { }
    }
}


fn main() { }

}



Overwriting /tmp/hourclock.rs


Hmm. Maybe I need opaque spec functions that just say what they do.

In [ ]:
def is_update(e):
    updates = {}
    todo = e
    while todo:
        if smt.is_and(e):
            todo.extend(e.children())
        

In [ ]:
"""
#[allow(unused_imports)]
use verus_builtin::*;
use verus_builtin_macros::*;
use vstd::prelude::*;
use vstd::{pervasive::*, *};

use verus_state_machines_macros::state_machine;
use verus_state_machines_macros::tokenized_state_machine;

verus! {
  state_machine!{
     Clock {
        fields {
        pub hr : int,
        }

        init!{
            initialize() {
            init hr = 1;
            }
        }
        transition!{
            HCnxt() {
                update hr = if pre.hr == 12 { 1 } else { pre.hr + 1 };
            }
        }
        #[invariant]
        pub fn is_typeof(&self) -> bool {
            self.hr >= 1 && self.hr <= 12
        }

        #[inductive(initialize)]
        fn initialize_inductive(post: Self) { }
       
        #[inductive(HCnxt)]
        fn HCnxt_inductive(pre: Self, post: Self) { }
    }
}
"""

def transition(mod, name):
    return f"""
    
    transition!{{
        {name}() {{
            update ...
        }}
    }}
    """

def invariant(mod, name):
    act = mod.action(name)
    State = kd.AStruct
    pre,post = smt.Consts("pre post" , State)
    vs = self.vars()
    vs1 = self.vars()
    body = smt.substitute(act, *[(v, getattr(pre, v.decl().name())) for v in vs], *[(v, getattr(post, v.decl().name())) for v in vs1])
    body.to_rust()
    return f"""
    #[invariant]
    pub fn {name}() -> bool {{
        {body}
    }}
    """


In [29]:
! /home/philip/vibe_coding/knuck_anal/knuckledragger/src/kdrag/solvers/verus/verus-x86-linux/verus /tmp/hourclock.rs

error: could not show invariant `is_typeof` on the `post` state
  --> /tmp/hourclock.rs:38:9
   |
38 | /         #[inductive(initialize)]
39 | |         fn initialize_inductive(post: Self) { }
   | |_______________________________________________^

error: could not show invariant `is_typeof` on the `post` state
  --> /tmp/hourclock.rs:41:9
   |
41 | /         #[inductive(HCnxt)]
42 | |         fn HCnxt_inductive(pre: Self, post: Self) { }
   | |_____________________________________________________^

verification results:: 2 verified, 2 errors
error: aborting due to 2 previous errors



In [15]:
import kdrag.solvers.verus as verus
verus.run_verus(["/tmp/hourclock.rs"]).stdout

CalledProcessError: Command '['/home/philip/vibe_coding/knuck_anal/knuckledragger/src/kdrag/solvers/verus/verus-x86-linux/verus', '/tmp/hourclock.rs']' returned non-zero exit status 1.

In [10]:
! /home/philip/vibe_coding/knuck_anal/knuckledragger/src/kdrag/solvers/verus/verus-x86-linux/verus /tmp/hourclock.rs

verification results:: 2 verified, 0 errors


https://github.com/tlaplus/tlaplus/blob/master/tlatools/org.lamport.tlatools/src/tla2sany/xml/sany.xsd

In [ ]:
import subprocess

xml = subprocess.run(["java", "-cp", "/home/philip/Downloads/tla2tools.jar", "tla2sany.xml.XMLExporter", "-o", "/tmp/HourClock.tla"], check=True, capture_output=True).stdout

import xml.etree.ElementTree as ET
mods = ET.fromstring(xml)
mods.tag
mods.attrib
mods[1][0]

ET.indent(mods, space="  ")
ET.tostring(mods, encoding="unicode")

# OpDeclNodeRef
# OpDeclNode
for e in mods.iter("OpDeclNode"): 
    print(ET.tostring(e, encoding="unicode"))


<OpDeclNode>
        <location>
          <column>
            <begin>10</begin>
            <end>11</end>
          </column>
          <line>
            <begin>4</begin>
            <end>4</end>
          </line>
          <filename>HourClock</filename>
        </location>
        <level>1</level>
        <uniquename>hr</uniquename>
        <arity>0</arity>
        <kind>3</kind>
      </OpDeclNode>
    


In [ ]:
import subprocess
from dataclasses import dataclass
import xml.etree.ElementTree as ET


@dataclass
class App:
    f: object
    args: list["App"]

    def __repr__(self):
        if not self.args:
            return f"{self.f}"
        else:
            return f"{self.f}({', '.join(map(str, self.args))})"


def tla_xml(filename):
    # https://github.com/tlaplus/tlaplus/blob/master/tlatools/org.lamport.tlatools/src/tla2sany/xml/sany.xsd
    return subprocess.run(
        [
            "java",
            "-cp",
            "/home/philip/Downloads/tla2tools.jar",
            "tla2sany.xml.XMLExporter",
            "-o",
            filename,
        ],
        check=True,
        capture_output=True,
    ).stdout


def tla_exprs(filename):
    mods = ET.fromstring(tla_xml(filename))

    by_uid = {}
    for entry in mods.find("context"):
        uid = entry.findtext("UID")
        node = next((c for c in entry if c.tag != "UID"), None)
        if uid is not None and node is not None:
            by_uid[uid] = node

    def ref_name(ref):
        uid = ref.findtext("UID")
        node = by_uid.get(uid)
        if node is None:
            return f"UID:{uid}"
        return node.findtext("uniquename") or node.tag

    def expr(node):
        if node.tag == "OpApplNode":
            ref = next(iter(node.find("operator")), None)
            f = ref_name(ref) if ref is not None else node.tag
            operands = node.find("operands")
            args = [] if operands is None else [expr(arg) for arg in operands]
            return App(f, args)
        if node.tag.endswith("Ref"):
            return App(ref_name(node), [])
        if node.tag == "NumeralNode":
            return App(int(node.findtext("IntValue")), [])
        return App(node.findtext("uniquename") or node.tag, [])

    res = {}
    for node in by_uid.values():
        if node.tag != "UserDefinedOpKind":
            continue
        if node.findtext("./location/filename") != mods.findtext("RootModule"):
            continue
        body = node.find("./body/*")
        if body is not None:
            res[node.findtext("uniquename")] = expr(body)
    return res


import z3


def to_z3(e: App, decls: dict[str, z3.FuncDeclRef], sort=None):
    if isinstance(e.f, int) and not e.args:
        assert sort is None or sort == z3.IntSort()
        return z3.IntVal(e.f)
    elif e.f in decls:
        f = decls[e.f]
        assert len(e.args) == f.arity()
        assert sort is None or sort == f.range()
        args = [to_z3(arg, decls, sort=f.domain(i)) for i, arg in enumerate(e.args)]
        return f(*args)
    elif e.f == "\\land":
        assert len(e.args) >= 2
        args = [to_z3(arg, decls, sort=z3.BoolSort()) for arg in e.args]
        return z3.And(*args)
    elif e.f == "\\lor":
        assert len(e.args) >= 2
        args = [to_z3(arg, decls, sort=z3.BoolSort()) for arg in e.args]
        return z3.Or(*args)
    elif e.f == "=":
        assert len(e.args) == 2
        args = [to_z3(arg, decls) for arg in e.args]
        return args[0] == args[1]
    elif e.f == "..":
        assert len(e.args) == 2
        args = [to_z3(arg, decls, sort=z3.IntSort()) for arg in e.args]
        x = z3.FreshConst(z3.IntSort(), prefix="x")
        return z3.Lambda(x, z3.And(args[0] <= x, x <= args[1]))
    elif e.f == "\\in":
        x = to_z3(e.args[0], decls)
        s = to_z3(e.args[1], decls, sort=z3.SetSort(x.sort()))
        return z3.IsMember(x, s)
    elif e.f == "'":
        assert len(e.args) == 1
        x = to_z3(e.args[0], decls, sort=sort)
        return z3.Const(x.decl().name() + "'", x.sort())
    elif e.f == "$IfThenElse":
        assert len(e.args) == 3
        cond = to_z3(e.args[0], decls, sort=z3.BoolSort())
        then = to_z3(e.args[1], decls, sort=sort)
        else_ = to_z3(e.args[2], decls, sort=sort)
        return z3.If(cond, then, else_)
    elif e.f == "+":
        assert len(e.args) == 2
        args = [to_z3(arg, decls, sort=sort) for arg in e.args]
        return args[0] + args[1]
    elif e.f == "$SquareAct":
        return z3.Const("$SQUAREACT TODO", z3.BoolSort())
    elif e.f == "[]":
        return z3.Const("[] TODO", z3.BoolSort())
    # elif sort is not None:
    #    # fallback to uninterprted function
    #    args = [to_z3(arg, decls) for arg in e.args]
    #    f = z3.Function(e.f, *[arg.sort() for arg in args], sort)
    #    return f(*args)
    else:
        raise ValueError(f"Cannot convert {e} to z3 without sort")


if __name__ == "__main__":
    exprs = tla_exprs("/tmp/HourClock.tla")
    decls = {"hr": z3.Function("hr", z3.IntSort())}
    for name, e in exprs.items():
        print(f"{name} = {e}")
        ze = to_z3(e, decls, sort=None)
        print(decls)
        decls[name] = z3.Function(name, ze.sort())
        print(ze)


"""
Notes:
I could take variable declarations and implicitly raise all definitions in the module to take those vars and their primed versions as explicit params.

Bidirectional type inference on the cheap.
Anonymous keyed records like tuples. That'd be kind of nice.


"""


# report



In [ ]:
def bisim(mod : Module, action_name, invs, ctx : pcode.BinaryContext, start, end, high_low):
    act = mod.action(action)
    inv = smt.And([act.predicate(i) for i in invs])
    mem = ctx.init_mem()
    simstates = ctx.execute_block(start, end)
    for i, simstate in enumerate(simstates):
        kd.prove(smt.Implies(smt.And(inv, smt.simstate), high_low(simstate.memstate)))
    

In [ ]:
class Bisim:
    mod : tla.Module
    actions
    def 


In [ ]:
class Report():

    def tla(self):
        # get from latex
    def bisim(self):
            
    def render(self):
        